In [1]:
import os
import pandas as pd
import tarfile
import gzip
from pathlib import Path
import requests
from io import StringIO, BytesIO
from IPython.display import display
import matplotlib.pyplot as plt

class GEODataAccess:
    def __init__(self, base_dir=None, series_id="GSE171485"):
        """
        Initialize GEO data access object.
        
        Args:
            base_dir (str): Local directory containing GEO files (None for remote access)
            series_id (str): GEO Series ID (e.g., "GSE171485")
        """
        self.base_dir = base_dir
        self.series_id = series_id

    def get_file_path(self, filename):
        """Get full path to a file"""
        if self.base_dir:
            return os.path.join(self.base_dir, filename)
        else:
            raise ValueError("Base directory is not set for local file access.")

    def load_file(self, filename):
        """Load a file based on its type"""
        file_path = self.get_file_path(filename)
        if filename.endswith('.soft'):
            return self._read_soft_file(file_path)
        elif filename.endswith('.tar'):
            return self._read_tar_file(file_path)
        elif filename.endswith('.tsv'):
            return self._read_tsv_file(file_path)
        elif filename.endswith('.csv'):
            return self._read_csv_file(file_path)
        elif filename.endswith('.txt'):
            return self._read_tsv_file(file_path)  # Assuming .txt files are tab-separated
        else:
            raise ValueError(f"Unsupported file type for file: {filename}")

    def _read_soft_file(self, file_path):
        """Read SOFT format file"""
        with open(file_path, 'r') as f:
            content = f.read()
        metadata = {}
        for line in content.split('\n'):
            if line.startswith('!'):
                key, value = line[1:].split('=', 1)
                metadata[key.strip()] = value.strip()
        return {'metadata': metadata}

    def _read_tar_file(self, file_path):
        """Read tar archive"""
        with tarfile.open(file_path, 'r:*') as tar:
            files = {}
            for member in tar.getmembers():
                if member.isfile():
                    files[member.name] = tar.extractfile(member).read().decode('utf-8')
            return files

    def _read_tsv_file(self, file_path):
        """Read TSV file"""
        return pd.read_csv(file_path, sep='\t', on_bad_lines='skip', engine='python')

    def _read_csv_file(self, file_path):
        """Read CSV file"""
        return pd.read_csv(file_path)

# Primer:
if __name__ == "__main__":
    # Set the base directory where the files are located
    base_dir = r"C:/Users/LENOVO/Documents/Uni/IJS/DataFile/"
    geo_access = GEODataAccess(base_dir=base_dir)

    # Load specific files
    family_soft = geo_access.load_file("GSE171485_family.soft")
    print("Family SOFT metadata:")
    print(family_soft['metadata'])

    family_xml = geo_access.load_file("GSE171485_family.xml.tar")
    print("\nFamily XML contents:")
    print(family_xml.keys())

    fpkm_data = geo_access.load_file("GSE171485_norm_counts_FPKM_GRCh38.p13_NCBI.tsv")
    print("\nFPKM data preview:")
    print(fpkm_data.head())

    tpm_data = geo_access.load_file("GSE171485_norm_counts_TPM_GRCh38.p13_NCBI.tsv")
    print("\nTPM data preview:")
    print(tpm_data.head())

    pdac_tissue_data = geo_access.load_file("GSE171485_PDAC-tissue-ExpressionMatrix.csv")
    print("\nPDAC tissue data preview:")
    print(pdac_tissue_data.head())

    raw_counts_data = geo_access.load_file("GSE171485_raw_counts_GRCh38.p13_NCBI.tsv")
    print("\nRaw counts data preview:")
    print(raw_counts_data.head())

    series_matrix = geo_access.load_file("GSE171485_series_matrix.txt")
    print("\nSeries matrix preview:")
    print(series_matrix.head())


Family SOFT metadata:
{'Database_name': 'Gene Expression Omnibus (GEO)', 'Database_institute': 'NCBI NLM NIH', 'Database_web_link': 'http://www.ncbi.nlm.nih.gov/geo', 'Database_email': 'geo@ncbi.nlm.nih.gov', 'Series_title': 'Identification and Functional Analysis of Novel Oncogenes in Pancreatic Ductal Adenocarcinoma', 'Series_geo_accession': 'GSE171485', 'Series_status': 'Public on Apr 06 2021', 'Series_submission_date': 'Apr 05 2021', 'Series_last_update_date': 'Nov 30 2021', 'Series_pubmed_id': '34789165', 'Series_summary': 'For identifying the specific expressed genes in PDAC, we constructed sequencing libraries from polyadenylated-RNA extracted from 6 PDAC specimens and 6 non-tumor adjacent tissues.', 'Series_overall_design': 'Examination of genes expression in PDAC and adjacent tissues.', 'Series_type': 'Expression profiling by high throughput sequencing', 'Series_contributor': 'Juehua,,Yu', 'Series_sample_id': 'GSM5530621', 'Series_contact_name': 'Hongjin,,Wu', 'Series_contact_

In [2]:
# Comparative table display function (polesna preglednost)
if __name__ == "__main__":
    # Set the base directory where the files are located
    base_dir = r"C:/Users/LENOVO/Documents/Uni/IJS/DataFile/"
    geo_access = GEODataAccess(base_dir=base_dir)

    # Function to display data in a clear table format
    def display_data_tables():
        print("Family SOFT Metadata:")
        metadata_df = pd.DataFrame(list(family_soft['metadata'].items()), columns=['Key', 'Value'])
        display(metadata_df)

        print("\nFPKM Data Preview:")
        display(fpkm_data.head())

        print("\nTPM Data Preview:")
        display(tpm_data.head())

        print("\nPDAC Tissue Data Preview:")
        display(pdac_tissue_data.head())

        print("\nRaw Counts Data Preview:")
        display(raw_counts_data.head())

        print("\nSeries Matrix Preview:")
        display(series_matrix.head())

    # Call the function to display tables
    display_data_tables()

Family SOFT Metadata:


,Key,Value
0,Database_name,Gene Expression Omnibus (GEO)
1,Database_institute,NCBI NLM NIH
2,Database_web_link,http://www.ncbi.nlm.nih.gov/geo
3,Database_email,geo@ncbi.nlm.nih.gov
4,Series_title,Identification and Functional Analysis of Nove...
...,...,...
67,Sample_library_strategy,RNA-Seq
68,Sample_relation,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...
69,Sample_supplementary_file_1,NONE
70,Sample_series_id,GSE171485



FPKM Data Preview:


,GeneID,GSM5226229,GSM5226230,GSM5226231,GSM5226232,GSM5226233,GSM5226234,GSM5530616,GSM5530617,GSM5530618,GSM5530619,GSM5530620,GSM5530621
0,100287102,0.10040,0.199,0.03024,0.1882,0.02647,0.1985,0.2016,0.1212,0.1792,0.8253,0.4523,0.05812
1,653635,7.12900,8.895,3.36000,12.3800,13.62000,12.1900,21.5100,14.3800,10.6100,43.5100,23.1100,6.16000
2,102466751,3.05000,3.454,0.00000,10.4500,11.58000,4.8220,4.8970,8.8340,0.0000,0.0000,7.8480,2.82400
3,107985730,0.07711,0.000,0.00000,0.0000,0.00000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1984,0.00000
4,100302278,0.00000,0.000,0.00000,0.0000,0.00000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00000



TPM Data Preview:


...

,gene_id,gene_short_name,tss_id,locus,PDAC-1,PDAC-2,PDAC-3,PDAC-4,PDAC-5,PDAC-6,CT-1,CT-2,CT-3,CT-4,CT-5,CT-6
0,ENSG00000000003,TSPAN6,-,X:99883666-99894988,3.397650,2.825978,4.028232,2.539709,3.929222,3.637880,4.283891,3.573119,4.788034,3.849344,3.573489,3.951680
1,ENSG00000000005,TNMD,-,X:99839798-99854882,0.072253,0.037555,1.002876,1.012851,0.260809,0.550408,5.645319,1.826536,0.077674,0.660918,0.271922,0.034571
2,ENSG00000000419,DPM1,-,20:49551403-49575092,4.378498,4.926889,4.616549,1.663841,4.608343,5.266389,5.184286,5.414795,5.005239,4.326951,4.849485,3.774648
3,ENSG00000000457,SCYL3,-,1:169818771-169863408,2.024935,2.074741,2.197670,1.352015,1.988266,1.680533,1.565279,1.951696,1.898304,1.773887,1.863808,1.670659
4,ENSG00000000460,C1orf112,-,1:169631244-169823221,1.595899,1.917478,1.399137,0.162467,0.658147,1.194877,0.873405,1.068689,0.688545,0.559568,0.731695,0.539084



Raw Counts Data Preview:


,GeneID,GSM5226229,GSM5226230,GSM5226231,GSM5226232,GSM5226233,GSM5226234,GSM5530616,GSM5530617,GSM5530618,GSM5530619,GSM5530620,GSM5530621
0,100287102,4,7,1,7,1,8,4,2,5,13,7,2
1,653635,304,335,119,493,551,526,457,254,317,734,383,227
2,102466751,5,5,0,16,18,8,4,6,0,0,5,4
3,107985730,1,0,0,0,0,0,0,0,0,0,1,0
4,100302278,0,0,0,0,0,0,0,0,0,0,0,0



Series Matrix Preview:


,!Series_title,Identification and Functional Analysis of Novel Oncogenes in Pancreatic Ductal Adenocarcinoma
0,!Series_geo_accession,GSE171485
1,!Series_status,Public on Apr 06 2021
2,!Series_submission_date,Apr 05 2021
3,!Series_last_update_date,Nov 30 2021
4,!Series_pubmed_id,34789165
